# 13 · LSTM

Task 16.7. One LSTM shared by all 50 stocks: a 60-day window of the 38 stock, market, macro and event features and the 6 calendar inputs, with
learned symbol and sector embeddings. Two heads give the probability of a rise and the log of realised volatility
for each horizon. Scalers are fitted on training rows only; the last 120 training days (after the gap) are held out
for early stopping. Refit yearly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.analysis.plots import (BLUE_CMAP, HORIZON_COLORS, MODEL_COLORS, TEXT_2, apply_style, bar_style,
                                dot_compare, figures_dir, save)
from src.config import Config
from src.evaluation.metrics import calibration_table, evaluate, join_labels, majority_guess, with_regimes
from src.models.data import label_frame, load_panel, regimes
from src.models.walkforward import run_or_load

cfg = Config.load()
apply_style()
pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)
val_start, val_end = cfg.period("validation")

In [ ]:
RETRAIN = False          # True retrains even when saved forecasts match the current settings
panel = load_panel(cfg)
labels = label_frame(panel)
fc = pd.concat([run_or_load(m, cfg, "validation", panel=panel, retrain=RETRAIN)
                for m in ["lstm", "base_rate", "hist_vol"]], ignore_index=True)
metrics = evaluate(fc, labels)
metrics[metrics.model == "lstm"].to_csv(figures_dir(cfg) / "lstm_validation.csv", index=False)
metrics[["model", "horizon_days", "n", "accuracy", "balanced_accuracy", "f1_up", "brier", "log_loss", "predicted_up",
         "vol_mae", "vol_rmse"]]

## Training curve (fold trained to the end of 2022)

In [ ]:
import json
fold = sorted((cfg.models_dir / "lstm" / "validation").glob("fold_*"))[0]
history = pd.DataFrame(json.loads((fold / "history.json").read_text()))
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(history.epoch, history.train_loss, marker="o", color=HORIZON_COLORS[1], label="training")
ax.plot(history.epoch, history.val_loss, marker="o", color=HORIZON_COLORS[10], label="early-stopping hold-out")
best = history.val_loss.idxmin()
ax.annotate("kept", (history.epoch[best], history.val_loss[best]), xytext=(0, -16), textcoords="offset points",
            ha="center", color=TEXT_2, fontsize=9)
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("LSTM training loss")
ax.legend()
save(fig, cfg, "13_lstm_training_curve")
plt.show()
history.round(4)

## Against the baselines

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
dot_compare(axes[0], metrics, "accuracy", ["lstm", "base_rate"], panel.horizons, reference=0.5)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title("Direction accuracy")
dot_compare(axes[1], metrics, "brier", ["lstm", "base_rate"], panel.horizons, fmt="{:.4f}")
axes[1].set_title("Brier score (lower is better)")
dot_compare(axes[2], metrics, "vol_mae", ["lstm", "hist_vol"], panel.horizons, fmt="{:.4f}")
axes[2].set_title("Volatility MAE (lower is better)")
fig.suptitle("LSTM vs baselines, validation 2023", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, cfg, "13_lstm_vs_baselines")
plt.show()

## Calibration: forecast probability vs how often the stock rose

In [ ]:
cal = calibration_table(join_labels(fc[fc.model == "lstm"], labels), bins=10)
fig, ax = plt.subplots(figsize=(5.2, 4.4))
for h in panel.horizons:
    g = cal[cal.horizon_days == h]
    ax.plot(g.p_mean, g.up_rate, marker="o", color=HORIZON_COLORS[h], label=f"{h}-day")
lims = [min(cal.p_mean.min(), cal.up_rate.min()) - 0.02, max(cal.p_mean.max(), cal.up_rate.max()) + 0.02]
ax.plot(lims, lims, color=TEXT_2, linewidth=1, linestyle="--")
ax.set_xlabel("mean forecast p_up (decile)")
ax.set_ylabel("actual share of up moves")
ax.set_title("LSTM calibration, validation 2023")
ax.legend()
save(fig, cfg, "13_lstm_calibration")
plt.show()
cal.pivot(index="bin", columns="horizon_days", values=["p_mean", "up_rate"]).round(3)

## Findings

1. **Training:** early stopping kept epoch 5 of 8 (hold-out loss 0.949); about 30 seconds per epoch on 12 CPU cores.
2. **Direction:** the probability head stays at the base rate (5-day p_up between 0.504 and 0.505 in every decile), so it predicts 'down' at 1 day and 'up' at 5 and 10 days for every stock-day: accuracy equals the base rate (53.0% / 56.2% / 60.6%) and balanced accuracy is 50%. Brier 0.2496 / 0.2495 / 0.2476 is slightly worse than the base rate's 0.2492 / 0.2485 / 0.2441.
3. **Volatility is the LSTM's useful output:** MAE 0.0069 / 0.0050 / 0.0045 vs historical volatility's 0.0088 / 0.0058 / 0.0050, i.e. about 22% / 14% / 10% lower error.
4. **The event inputs move the volatility forecast the right way, but not far enough:** on the 190 results days of 2023 the mean 1-day forecast rose from 0.76% (before the event inputs) to 0.99% and the error fell from 0.0160 to 0.0152, while the realised move averaged 2.13%. Error on other days is unchanged (0.0067 to 0.0066).
5. **Next (tune on validation only):** a direction-only loss, a lower learning rate and fewer inputs, to see whether the probability head can move away from the base rate.